In [ ]:
# ============================================================
# Final Project Details
# ============================================================

# Project Title - Crop Disease Detection Using CNN and Transfer Learning
  ### ICTAK Academy — Certified Specialist in ML and AI
  ### Student: Arun Krishnan G.

In [ ]:
# ============================================================
# STEP 1: Import All Required Libraries
# ============================================================

import os                          # For folder and file operations
import random                      # For selecting random sample images
import warnings                    # For hiding unnecessary warning messages
import numpy as np                 # For numerical operations on image arrays
import matplotlib.pyplot as plt    # For plotting charts and displaying images
import seaborn as sns              # For enhanced statistical visualisations
import tensorflow as tf            # Main deep learning framework
from tensorflow.keras.preprocessing.image import (
    ImageDataGenerator,            # For loading and augmenting images
    load_img,                      # For loading a single image
    img_to_array                   # For converting image to numpy array
)
from tensorflow.keras.applications import MobileNetV2   # Pre-trained model
from tensorflow.keras import layers, models             # For building our model
from tensorflow.keras.optimizers import Adam            # Optimizer for training
from tensorflow.keras.callbacks import (
    EarlyStopping,                 # Stops training when model stops improving
    ModelCheckpoint,               # Saves the best model automatically
    ReduceLROnPlateau              # Reduces learning rate when stuck
)
from sklearn.metrics import (
    classification_report,         # Detailed accuracy per class
    confusion_matrix               # Shows where model gets confused
)

warnings.filterwarnings('ignore')

print("=" * 50)
print("All Libraries Imported Successfully!")
print(f"TensorFlow Version : {tf.__version__}")
print(f"GPU Available      : {len(tf.config.list_physical_devices('GPU')) > 0}")
print("=" * 50)

In [ ]:
# ============================================================
# Find the exact folder path for the Datasets used
# ============================================================
import os

base = '/kaggle/input/datasets'

print("=== LEAF DATASET FULL STRUCTURE ===")
leaf = f'{base}/dev523/leaf-disease-detection-dataset'
for root, dirs, files in os.walk(leaf):
    level = root.replace(leaf, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent} {os.path.basename(root)}/")
    if level >= 3:
        dirs.clear()

print("\n=== RICE DATASET FULL STRUCTURE ===")
rice = f'{base}/minhhuy2810/rice-diseases-image-dataset'
for root, dirs, files in os.walk(rice):
    level = root.replace(rice, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent} {os.path.basename(root)}/")
    if level >= 3:
        dirs.clear()

In [ ]:
import os

# ============================================================
# STEP 2: Set Correct Dataset Paths and Verify
# ============================================================

# Leaf Disease Dataset
LEAF_TRAIN = '/kaggle/input/datasets/dev523/leaf-disease-detection-dataset/dataset/train'
LEAF_VALID = '/kaggle/input/datasets/dev523/leaf-disease-detection-dataset/dataset/test'

# Rice Disease Dataset — using RiceDiseaseDataset as it has train/validation split
RICE_TRAIN = '/kaggle/input/datasets/minhhuy2810/rice-diseases-image-dataset/RiceDiseaseDataset/train'
RICE_VALID = '/kaggle/input/datasets/minhhuy2810/rice-diseases-image-dataset/RiceDiseaseDataset/validation'

# Verify all paths
print("Checking all dataset paths...")
print(f"Leaf Train : {os.path.exists(LEAF_TRAIN)}")
print(f"Leaf Valid : {os.path.exists(LEAF_VALID)}")
print(f"Rice Train : {os.path.exists(RICE_TRAIN)}")
print(f"Rice Valid : {os.path.exists(RICE_VALID)}")

# Count classes
leaf_classes = sorted(os.listdir(LEAF_TRAIN))
rice_classes = sorted(os.listdir(RICE_TRAIN))

print(f"\n Dataset Summary:")
print(f"   Leaf Disease classes : {len(leaf_classes)}")
print(f"   Rice Disease classes : {len(rice_classes)}")
print(f"   Total classes        : {len(leaf_classes) + len(rice_classes)}")

# Count images
leaf_train_count = sum(len(os.listdir(os.path.join(LEAF_TRAIN, c))) for c in leaf_classes)
leaf_valid_count = sum(len(os.listdir(os.path.join(LEAF_VALID, c))) for c in leaf_classes)
rice_train_count = sum(len(os.listdir(os.path.join(RICE_TRAIN, c))) for c in rice_classes)
rice_valid_count = sum(len(os.listdir(os.path.join(RICE_VALID, c))) for c in rice_classes)

print(f"\n   Leaf Train Images  : {leaf_train_count:,}")
print(f"   Leaf Valid Images  : {leaf_valid_count:,}")
print(f"   Rice Train Images  : {rice_train_count:,}")
print(f"   Rice Valid Images  : {rice_valid_count:,}")
print(f"\n   Grand Total        : {leaf_train_count + leaf_valid_count + rice_train_count + rice_valid_count:,}")

In [ ]:
# ============================================================
# STEP 3: Display Sample Leaf Images from Dataset
# ============================================================

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
fig.suptitle('Sample Leaf Images — Disease Classes',
             fontsize=16, fontweight='bold', color='darkgreen')

selected = random.sample(leaf_classes, 12)

for i, ax in enumerate(axes.flat):
    folder   = os.path.join(LEAF_TRAIN, selected[i])
    img_file = random.choice(os.listdir(folder))
    img      = load_img(os.path.join(folder, img_file),
                        target_size=(224, 224))
    ax.imshow(img)
    ax.set_title(selected[i].replace('___', '\n').replace('_', ' ')[:25],
                 fontsize=8, color='darkgreen', fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.show()
print("Sample images displayed successfully!")

In [ ]:
# ============================================================
# STEP 4: Plot Class Distribution Chart
# ============================================================

# Count images per class
leaf_counts = [len(os.listdir(os.path.join(LEAF_TRAIN, c))) for c in leaf_classes]
rice_counts = [len(os.listdir(os.path.join(RICE_TRAIN, c))) for c in rice_classes]

# Combine all classes and counts
all_classes = leaf_classes + [f"Rice_{c}" for c in rice_classes]
all_counts  = leaf_counts + rice_counts

# Plot
fig, axes = plt.subplots(2, 1, figsize=(22, 12))

# --- Chart 1: Leaf Disease Classes ---
axes[0].bar(range(len(leaf_classes)), leaf_counts,
            color='green', alpha=0.75, edgecolor='darkgreen')
axes[0].set_xticks(range(len(leaf_classes)))
axes[0].set_xticklabels(
    [c.replace('___', '\n').replace('_', ' ') for c in leaf_classes],
    rotation=90, fontsize=7)
axes[0].set_title('Leaf Disease Dataset — Images per Class',
                  fontsize=13, fontweight='bold', color='darkgreen')
axes[0].set_ylabel('Number of Images')
axes[0].grid(axis='y', alpha=0.3)

# --- Chart 2: Rice Disease Classes ---
axes[1].bar(range(len(rice_classes)), rice_counts,
            color='goldenrod', alpha=0.75, edgecolor='darkgoldenrod')
axes[1].set_xticks(range(len(rice_classes)))
axes[1].set_xticklabels(rice_classes, rotation=0, fontsize=10)
axes[1].set_title('Rice Disease Dataset — Images per Class',
                  fontsize=13, fontweight='bold', color='goldenrod')
axes[1].set_ylabel('Number of Images')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# Summary
print("=" * 50)
print(" Dataset Distribution Summary:")
print(f"   Leaf — Max images in one class : {max(leaf_counts):,}")
print(f"   Leaf — Min images in one class : {min(leaf_counts):,}")
print(f"   Leaf — Avg images per class    : {int(sum(leaf_counts)/len(leaf_counts)):,}")
print(f"   Rice — Max images in one class : {max(rice_counts):,}")
print(f"   Rice — Min images in one class : {min(rice_counts):,}")
print(f"   Rice — Avg images per class    : {int(sum(rice_counts)/len(rice_counts)):,}")
print("=" * 50)
print("Class distribution chart completed!")

In [ ]:
# ============================================================
# STEP 5: Combine Both Datasets into One Folder
# ============================================================

import shutil

# Source paths — confirmed last week
LEAF_TRAIN = '/kaggle/input/datasets/dev523/leaf-disease-detection-dataset/dataset/train'
LEAF_VALID = '/kaggle/input/datasets/dev523/leaf-disease-detection-dataset/dataset/test'
RICE_TRAIN = '/kaggle/input/datasets/minhhuy2810/rice-diseases-image-dataset/RiceDiseaseDataset/train'
RICE_VALID = '/kaggle/input/datasets/minhhuy2810/rice-diseases-image-dataset/RiceDiseaseDataset/validation'

# Destination paths
COMBINED_TRAIN = '/kaggle/working/combined/train'
COMBINED_VALID  = '/kaggle/working/combined/valid'

os.makedirs(COMBINED_TRAIN, exist_ok=True)
os.makedirs(COMBINED_VALID,  exist_ok=True)

# --- Copy Leaf Train folders ---
print("Copying Leaf Disease training folders...")
for folder in os.listdir(LEAF_TRAIN):
    src = os.path.join(LEAF_TRAIN, folder)
    dst = os.path.join(COMBINED_TRAIN, folder)
    if not os.path.exists(dst):
        shutil.copytree(src, dst)
print(f"Leaf train folders copied : {len(os.listdir(COMBINED_TRAIN))}")

# --- Copy Leaf Valid folders ---
print("\nCopying Leaf Disease validation folders...")
for folder in os.listdir(LEAF_VALID):
    src = os.path.join(LEAF_VALID, folder)
    dst = os.path.join(COMBINED_VALID, folder)
    if not os.path.exists(dst):
        shutil.copytree(src, dst)
print(f"Leaf valid folders copied : {len(os.listdir(COMBINED_VALID))}")

# --- Copy Rice Train folders ---
print("\nCopying Rice Disease training folders...")
for folder in os.listdir(RICE_TRAIN):
    src        = os.path.join(RICE_TRAIN, folder)
    # Add Rice_ prefix to avoid name clash with Leaf healthy folders
    dst        = os.path.join(COMBINED_TRAIN, f"Rice_{folder}")
    if not os.path.exists(dst):
        shutil.copytree(src, dst)

# --- Copy Rice Valid folders ---
print("Copying Rice Disease validation folders...")
for folder in os.listdir(RICE_VALID):
    src = os.path.join(RICE_VALID, folder)
    dst = os.path.join(COMBINED_VALID, f"Rice_{folder}")
    if not os.path.exists(dst):
        shutil.copytree(src, dst)

# --- Final Summary ---
all_classes = sorted(os.listdir(COMBINED_TRAIN))
total_train = sum(len(os.listdir(os.path.join(COMBINED_TRAIN, c))) for c in all_classes)
total_valid = sum(len(os.listdir(os.path.join(COMBINED_VALID, c))) for c in all_classes)

print("\n" + "=" * 55)
print("COMBINED DATASET READY!")
print(f"   Total Classes          : {len(all_classes)}")
print(f"   Total Train Images     : {total_train:,}")
print(f"   Total Valid Images     : {total_valid:,}")
print(f"   Grand Total            : {total_train + total_valid:,}")
print("=" * 55)

In [ ]:
# ============================================================
# STEP 6: Define Hyperparameters
# (Research-based starting values)
# ============================================================

IMG_SIZE      = 224    # MobileNetV2 standard input size
BATCH_SIZE    = 32     # Most stable batch size for image classification
EPOCHS        = 15     # Sufficient for Transfer Learning
LEARNING_RATE = 0.001  # Adam optimizer default — proven starting point
DROPOUT_RATE  = 0.3    # Prevents overfitting — standard starting value
DENSE_UNITS   = 256    # Balanced capacity for classification layer
NUM_CLASSES   = len(os.listdir(COMBINED_TRAIN))

print("=" * 55)
print("Hyperparameters Defined!")
print(f"   Image Size        : {IMG_SIZE} x {IMG_SIZE}")
print(f"   Batch Size        : {BATCH_SIZE}")
print(f"   Epochs            : {EPOCHS}")
print(f"   Learning Rate     : {LEARNING_RATE}")
print(f"   Dropout Rate      : {DROPOUT_RATE}")
print(f"   Dense Units       : {DENSE_UNITS}")
print(f"   Number of Classes : {NUM_CLASSES}")
print("=" * 55)

In [ ]:
# ============================================================
# STEP 7: Image Preprocessing and Data Augmentation
# ============================================================

from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Update paths to use combined dataset
TRAIN_PATH = COMBINED_TRAIN
VALID_PATH  = COMBINED_VALID

# Get all classes from combined dataset
classes    = sorted(os.listdir(TRAIN_PATH))
NUM_CLASSES = len(classes)

print(f"Total classes to train on: {NUM_CLASSES}")

# --- Training Generator WITH Augmentation ---
train_datagen = ImageDataGenerator(
    rescale            = 1./255,  # Normalise pixels from 0-255 to 0-1
    rotation_range     = 20,      # Randomly rotate image up to 20 degrees
    zoom_range         = 0.2,     # Randomly zoom in up to 20%
    horizontal_flip    = True,    # Randomly flip image left to right
    width_shift_range  = 0.1,     # Shift image left or right by 10%
    height_shift_range = 0.1,     # Shift image up or down by 10%
    shear_range        = 0.1,     # Slightly distort image shape
    fill_mode          = 'nearest' # Fill empty pixels after transformation
)

# --- Validation Generator — ONLY Rescale, No Augmentation ---
valid_datagen = ImageDataGenerator(
    rescale = 1./255   # Only normalise — no augmentation on validation
)

# --- Load Training Images ---
train_generator = train_datagen.flow_from_directory(
    TRAIN_PATH,
    target_size  = (IMG_SIZE, IMG_SIZE),  # Resize all to 224x224
    batch_size   = BATCH_SIZE,            # Load 32 images at a time
    class_mode   = 'categorical',         # Multiple disease classes
    shuffle      = True                   # Shuffle for better training
)

# --- Load Validation Images ---
valid_generator = valid_datagen.flow_from_directory(
    VALID_PATH,
    target_size  = (IMG_SIZE, IMG_SIZE),
    batch_size   = BATCH_SIZE,
    class_mode   = 'categorical',
    shuffle      = False                  # Keep order for accurate evaluation
)

print("\n" + "=" * 55)
print("Preprocessing Complete!")
print(f"   Training samples   : {train_generator.samples:,}")
print(f"   Validation samples : {valid_generator.samples:,}")
print(f"   Number of classes  : {train_generator.num_classes}")
print(f"   Image size         : {IMG_SIZE} x {IMG_SIZE}")
print(f"   Batch size         : {BATCH_SIZE}")
print("=" * 55)

In [ ]:
# ============================================================
# STEP 8: Load MobileNetV2 and Build Complete Model
# ============================================================

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam

# --- Step 1: Load Pre-trained MobileNetV2 ---
base_model = MobileNetV2(
    input_shape = (IMG_SIZE, IMG_SIZE, 3),
    # 224x224 image with 3 colour channels (RGB)
    # Must match the size MobileNetV2 was originally trained on

    include_top = False,
    # Remove the original classification head
    # We will add our own for 42 plant disease classes

    weights = 'imagenet'
    # Use weights pre-trained on ImageNet
    # ImageNet = 1.4 million images, 1000 categories
    # This gives our model powerful visual knowledge from day one
)

# --- Step 2: Freeze Base Model ---
base_model.trainable = False
# We do NOT change MobileNetV2 weights during training
# We only train our new custom layers on top
# This is the core concept of Transfer Learning

# --- Step 3: Build Complete Model ---
model = models.Sequential([

    base_model,
    # MobileNetV2 as feature extractor
    # Converts 224x224 image into rich feature maps

    layers.GlobalAveragePooling2D(),
    # Converts feature maps into a single flat vector
    # Better than Flatten() for Transfer Learning
    # Reduces parameters and prevents overfitting

    layers.Dense(DENSE_UNITS, activation='relu'),
    # 256 neurons — our custom learning layer
    # ReLU removes negative values — helps learn complex patterns

    layers.Dropout(DROPOUT_RATE),
    # Randomly switches off 30% of neurons during training
    # Forces model to learn multiple pathways
    # Prevents overfitting — makes model more general

    layers.Dense(NUM_CLASSES, activation='softmax')
    # Output layer — one neuron per disease class (42 total)
    # Softmax converts outputs to probabilities summing to 1
    # e.g. 85% Tomato Early Blight, 10% Late Blight, 5% other
])

# --- Step 4: Compile Model ---
model.compile(
    optimizer = Adam(learning_rate=LEARNING_RATE),
    # Adam adapts learning rate automatically
    # Best choice for image classification tasks

    loss = 'categorical_crossentropy',
    # Standard loss function for multi-class classification
    # Measures how wrong the model predictions are

    metrics = ['accuracy']
    # Track accuracy during training and validation
)

print("=" * 55)
print("MobileNetV2 Loaded Successfully!")
print(f"   Base model layers    : {len(base_model.layers)}")
print(f"   Base model frozen    : Yes — Transfer Learning ready")
print(f"   Output classes       : {NUM_CLASSES}")
print("\nFull Model Built and Compiled!")
print("=" * 55)
model.summary()

In [ ]:
# ============================================================
# STEP 9: Define Callbacks
# ============================================================

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau
)

callbacks = [

    EarlyStopping(
        monitor             = 'val_accuracy',
        # Watch validation accuracy — not training accuracy
        # We care about how well model works on unseen images

        patience            = 5,
        # Stop if no improvement for 5 epochs in a row
        # Saves time and prevents overfitting

        restore_best_weights = True,
        # Go back to the best weights automatically
        # Even if last epoch was worse — we keep the best

        verbose             = 1
    ),

    ModelCheckpoint(
        'best_model.keras',
        # Save file name

        monitor             = 'val_accuracy',
        # Save only when validation accuracy improves

        save_best_only      = True,
        # Keep only the best version — saves storage

        verbose             = 1
    ),

    ReduceLROnPlateau(
        monitor             = 'val_loss',
        # Watch validation loss

        factor              = 0.2,
        # Reduce learning rate by 80% when triggered
        # From 0.001 to 0.0002 for example

        patience            = 3,
        # Wait 3 epochs before reducing
        # Gives model a chance to improve first

        min_lr              = 1e-7,
        # Never go below this learning rate
        # Prevents learning rate becoming zero

        verbose             = 1
    )
]

print("=" * 55)
print("Callbacks Configured!")
print("   # EarlyStopping      — stops when no improvement")
print("   # ModelCheckpoint    — saves best model")
print("   # ReduceLROnPlateau  — reduces LR when stuck")
print("=" * 55)

In [ ]:
# ======================================================================
# Did a second round of training, as last time forgot to save the model
# ======================================================================

# ======================================================================
# STEP 10: Train the Model and Save Immediately - Model 1
# ======================================================================

print("Starting Model 1 Training...")
print(f"   Architecture   : MobileNetV2")
print(f"   Learning Rate  : {LEARNING_RATE}")
print(f"   Dropout Rate   : {DROPOUT_RATE}")
print(f"   Dense Units    : {DENSE_UNITS}")
print(f"   Epochs         : {EPOCHS}")
print(f"   Training on    : {train_generator.samples:,} images")
print(f"   Validating on  : {valid_generator.samples:,} images\n")

history_model1 = model.fit(
    train_generator,
    epochs          = EPOCHS,
    validation_data = valid_generator,
    callbacks       = callbacks,
    verbose         = 1
)

# ============================================================
# SAVE IMMEDIATELY — DO NOT RUN SEPARATELY!
# ============================================================
print("\nSaving Model 1")
model.save('/kaggle/working/model1_mobilenetv2.keras')

# Verify saved correctly
import os
size = os.path.getsize('/kaggle/working/model1_mobilenetv2.keras') / (1024*1024)
print(f"Model 1 saved successfully!")
print(f"File size : {size:.2f} MB")

# Save training history for later use in charts
import numpy as np
np.save('/kaggle/working/history_model1.npy', history_model1.history)
print("Training history saved!")

# Print final results
print("\n" + "=" * 55)
print("MODEL 1 TRAINING COMPLETE!")
print(f"   Best Training Accuracy   : {max(history_model1.history['accuracy'])*100:.2f}%")
print(f"   Best Validation Accuracy : {max(history_model1.history['val_accuracy'])*100:.2f}%")
print(f"   Total Epochs Run         : {len(history_model1.history['accuracy'])}")
print("=" * 55)
print("\nModel - model1_mobilenetv2.keras, is ready to be downloaded from Output tab!")

In [ ]:
# ============================================================
# STEP 11: FIND THE CROP DISEASE MODEL 1 DATASET PATH
# ============================================================

import os

base = '/kaggle/input/datasets'
print("=== CONTENTS OF DATASETS FOLDER ===")
for item in os.listdir(base):
    print(f" {item}")
    subfolder = os.path.join(base, item)
    for sub in os.listdir(subfolder):
        print(f"    {sub}")
        subpath = os.path.join(subfolder, sub)
        if os.path.isdir(subpath):
            for f in os.listdir(subpath):
                print(f"      📄 {f}")

In [ ]:
# ============================================================
# STEP 12: LOAD SAVED MODEL 1
# ============================================================
import tensorflow as tf
import numpy as np

# Load Model
model = tf.keras.models.load_model(
    '/kaggle/input/datasets/arunkrisg/crop-disease-model-1/model1_mobilenetv2.keras'
)

print("=" * 55)
print("Model 1 Loaded Successfully!")
print(f"   Architecture : MobileNetV2")
print(f"   Total layers : {len(model.layers)}")
print(f"   Input shape  : {model.input_shape}")
print(f"   Output shape : {model.output_shape}")
print("=" * 55)

# Load Training History
history_data = np.load(
    '/kaggle/input/datasets/arunkrisg/crop-disease-model-1/history_model1.npy',
    allow_pickle=True
).item()

print("Training history loaded!")
print(f"   Epochs completed    : {len(history_data['accuracy'])}")
print(f"   Best Train Accuracy : {max(history_data['accuracy'])*100:.2f}%")
print(f"   Best Val Accuracy   : {max(history_data['val_accuracy'])*100:.2f}%")

In [ ]:
# ============================================================
# STEP 13: Plot Accuracy and Loss Charts
# ============================================================
import matplotlib.pyplot as plt

epochs_range = range(1, len(history_data['accuracy']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Model 1 — MobileNetV2 Training Results',
             fontsize=16, fontweight='bold', color='darkgreen')

# --- Chart 1: Accuracy ---
axes[0].plot(epochs_range, 
             [a*100 for a in history_data['accuracy']],
             'o-', color='green', linewidth=2, label='Training Accuracy')
axes[0].plot(epochs_range,
             [a*100 for a in history_data['val_accuracy']],
             's--', color='orange', linewidth=2, label='Validation Accuracy')
axes[0].set_title('Training vs Validation Accuracy',
                  fontweight='bold', color='darkgreen')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy (%)')
axes[0].legend()
axes[0].grid(alpha=0.3)
axes[0].set_xticks(epochs_range)

# --- Chart 2: Loss ---
axes[1].plot(epochs_range,
             history_data['loss'],
             'o-', color='red', linewidth=2, label='Training Loss')
axes[1].plot(epochs_range,
             history_data['val_loss'],
             's--', color='blue', linewidth=2, label='Validation Loss')
axes[1].set_title('Training vs Validation Loss',
                  fontweight='bold', color='darkred')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(alpha=0.3)
axes[1].set_xticks(epochs_range)

plt.tight_layout()
plt.savefig('/kaggle/working/training_charts.png',
            dpi=150, bbox_inches='tight')
plt.show()

print("Charts saved as training_charts.png!")
print(f"\nSummary:")
print(f"   Start Train Accuracy : {history_data['accuracy'][0]*100:.2f}%")
print(f"   Final Train Accuracy : {history_data['accuracy'][-1]*100:.2f}%")
print(f"   Start Val Accuracy   : {history_data['val_accuracy'][0]*100:.2f}%")
print(f"   Final Val Accuracy   : {history_data['val_accuracy'][-1]*100:.2f}%")
print(f"   Start Loss           : {history_data['loss'][0]:.4f}")
print(f"   Final Loss           : {history_data['loss'][-1]:.4f}")

In [ ]:
# ============================================================
# STEP 14: Confusion Matrix
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# Get class names
class_names = list(valid_generator.class_indices.keys())

# Get predictions
print("Getting model predictions on validation data...")
predictions = model.predict(valid_generator, verbose=1)
predicted_classes = np.argmax(predictions, axis=1)
true_classes = valid_generator.classes

# --- Classification Report ---
print("\nClassification Report:")
report = classification_report(true_classes, predicted_classes,
                               target_names=class_names,
                               digits=2)
print(report)

# --- Confusion Matrix ---
cm = confusion_matrix(true_classes, predicted_classes)

# Plot — Full confusion matrix
plt.figure(figsize=(24, 20))
sns.heatmap(cm,
            annot=True,
            fmt='d',
            cmap='Greens',
            xticklabels=[c.replace('___', '\n').replace('_', ' ')[:15]
                         for c in class_names],
            yticklabels=[c.replace('___', '\n').replace('_', ' ')[:15]
                         for c in class_names],
            linewidths=0.5)

plt.title('Confusion Matrix — Crop Disease Detection Model',
          fontsize=18, fontweight='bold', pad=20)
plt.xlabel('Predicted Disease', fontsize=13)
plt.ylabel('Actual Disease', fontsize=13)
plt.xticks(fontsize=7, rotation=90)
plt.yticks(fontsize=7, rotation=0)
plt.tight_layout()
plt.savefig('/kaggle/working/confusion_matrix.png',
            dpi=150, bbox_inches='tight')
plt.show()

# --- Key Metrics ---
overall_accuracy = np.trace(cm) / np.sum(cm) * 100
print(f"\n{'='*55}")
print(f"Confusion Matrix Complete!")
print(f"   Overall Accuracy  : {overall_accuracy:.2f}%")
print(f"   Total Classes     : {len(class_names)}")
print(f"   Total Predictions : {len(predicted_classes):,}")
print(f"{'='*55}")

In [ ]:
# ============================================================
# Apply Fix and Save the Model
# ============================================================
import tensorflow as tf
import os

# Load old model
old_model = tf.keras.models.load_model(
    '/kaggle/input/datasets/arunkrisg/crop-disease-model-1/model1_mobilenetv2.keras'
)

# Build using Functional API
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers

base = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights=None
)
base.trainable = False

# Functional API — fixes compatibility issue!
inputs  = tf.keras.Input(shape=(224, 224, 3))
x       = base(inputs, training=False)
x       = layers.GlobalAveragePooling2D()(x)
x       = layers.Dense(256, activation='relu')(x)
x       = layers.Dropout(0.3)(x)
outputs = layers.Dense(42, activation='softmax')(x)

new_model = tf.keras.Model(inputs=inputs, outputs=outputs)

# Copy weights from old model
new_model.set_weights(old_model.get_weights())

new_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Save
new_model.save('/kaggle/working/model_final.keras')
print("Final compatible model saved!")

size = os.path.getsize('/kaggle/working/model_final.keras')/(1024*1024)
print(f"Size: {size:.2f} MB")
print("Download model_final.keras from Output tab")